In [ ]:
# ============================================================================
# INSTALLATION & SETUP
# ============================================================================

!pip install -q openai python-docx pillow sentence-transformers scikit-learn
!pip install -q PyMuPDF  # fitz for PDF processing
!pip install -q docling  # Advanced document parsing
!pip install -q textstat

print("✅ All packages installed successfully!")

# ============================================================================
# IMPORTS
# ============================================================================

import os
import json
import base64
import re
from typing import Dict, List, Optional, Tuple
from io import BytesIO
from datetime import datetime

# Security: Handle Environment Variables
from google.colab import userdata

# File processing
import fitz  # PyMuPDF
import docx
from PIL import Image

# Docling for advanced parsing
try:
    from docling.document_converter import DocumentConverter
    from docling.datamodel.base_models import InputFormat
    DOCLING_AVAILABLE = True
except ImportError:
    print("⚠️ Docling not available, using fallback methods")
    DOCLING_AVAILABLE = False

# ML
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import textstat
import torch # Added import for torch

# AI Detection
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# API client
from openai import OpenAI

# Colab
from google.colab import files

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

PROVIDER = "openrouter"  # "groq" or "openrouter"

# ------------------------------------------------------------------
# 🔒 SECURE KEY RETRIEVAL (التعديل الجديد)
# ------------------------------------------------------------------
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')
    print("🔐 API Keys retrieved from Colab Secrets.")

except Exception as e:

    GROQ_API_KEY = os.getenv("GROQ_API_KEY")
    OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
    print(f"⚠️ Note: Could not retrieve keys from Colab Secrets. Checking OS Environment.")

if PROVIDER == "groq" and not GROQ_API_KEY:
    raise ValueError("🚨 CRITICAL: GROQ_API_KEY is missing! Please add it to Colab Secrets.")
if PROVIDER == "openrouter" and not OPENROUTER_API_KEY:
    raise ValueError("🚨 CRITICAL: OPENROUTER_API_KEY is missing! Please add it to Colab Secrets.")

print("🔐 API Keys loaded securely.")

# ============================================================================
# CLIENT + MODELS
# ============================================================================

if  PROVIDER == "openrouter":
    print("🎯 Using OpenRouter backend")

    client = OpenAI(
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
    )

    PARSING_MODEL = "meta-llama/llama-3.3-70b-instruct"
    PARSING_FALLBACK = "meta-llama/llama-3.1-70b-instruct"
    ATS_MODEL = "meta-llama/llama-3.3-70b-instruct"
    SCORING_MODEL = "meta-llama/llama-3.3-70b-instruct"
    OCR_MODEL = "meta-llama/llama-3.2-90b-vision-preview"

elif PROVIDER == "groq":
    print("⚡ Using Groq backend")
    # Assuming Groq client usage, usually OpenAI compatible or specific library
    client = OpenAI(
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1"
    )

    PARSING_MODEL = "llama-3.3-70b-versatile"
    PARSING_FALLBACK = "llama-3.1-70b-versatile"
    ATS_MODEL = "llama-3.3-70b-versatile"
    SCORING_MODEL = "llama-3.3-70b-versatile"
    OCR_MODEL = "llama-3.2-90b-vision-preview"

else:
    raise ValueError("❌ Invalid PROVIDER selected")

# Load embedding model
print("🔄 Loading embedding model...")
embedding_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
print("✅ Embedding model loaded!")

In [ ]:
# ============================================================================
# 🕵️‍♂️ AI CONTENT DETECTOR CLASS
# ============================================================================

class AIDetector:
    """
    Detects AI-generated content using Perplexity and Burstiness analysis.
    Uses GPT-2 (small) to calculate likelihood of text sequences.
    """
    def __init__(self):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🕵️‍♂️ Initializing AI Detector on {self.device}...")

        # Load GPT-2 model (Standard for perplexity checks)
        model_id = "gpt2"
        self.model = GPT2LMHeadModel.from_pretrained(model_id).to(self.device)
        self.tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
        self.max_length = self.model.config.n_positions

    def calculate_perplexity(self, text):
        """Calculates perplexity of a text."""
        encodings = self.tokenizer(text, return_tensors="pt")
        input_ids = encodings.input_ids.to(self.device)

        with torch.no_grad():
            outputs = self.model(input_ids, labels=input_ids)
            loss = outputs.loss
            perplexity = torch.exp(loss)

        return perplexity.item()

    def analyze_text(self, text):
        """
        Analyzes text for AI probability based on Perplexity & Burstiness.
        """
        # 1. Clean text
        text = re.sub(r'\n+', ' ', text).strip()
        if not text: return {"is_ai": False, "confidence": 0, "details": "Empty text"}

        # 2. Split into sentences for Burstiness
        sentences = [s.strip() for s in text.split('.') if len(s.strip()) > 10]
        if len(sentences) < 3:
            # Not enough text for reliable analysis
            overall_ppl = self.calculate_perplexity(text)
            return {
                "score": 50,
                "verdict": "Insufficient Data",
                "perplexity": round(overall_ppl, 2),
                "burstiness": 0
            }

        # 3. Calculate Perplexity for each sentence
        perplexities = []
        for sentence in sentences[:20]: # Analyze first 20 sentences to save time
            try:
                ppl = self.calculate_perplexity(sentence)
                perplexities.append(ppl)
            except:
                continue

        if not perplexities:
            return {"score": 0, "verdict": "Error", "perplexity": 0, "burstiness": 0}

        # 4. Metrics
        avg_perplexity = np.mean(perplexities)
        burstiness = np.std(perplexities) # Standard Deviation represents burstiness

        # 5. Logic / Heuristics (Thresholds need tuning, these are standard baselines)
        # AI Text: Low Perplexity (Predictable) + Low Burstiness (Monotone)
        # Human Text: High Perplexity (Creative) + High Burstiness (Varied)

        ai_score = 0

        # Perplexity Logic
        if avg_perplexity < 40: ai_score += 60      # Very predictable
        elif avg_perplexity < 70: ai_score += 40    # Somewhat predictable
        elif avg_perplexity < 100: ai_score += 20   # Balanced

        # Burstiness Logic
        if burstiness < 15: ai_score += 40          # Very monotone
        elif burstiness < 30: ai_score += 20        # Somewhat monotone

        # Cap score
        ai_score = min(ai_score, 99)

        # Verdict
        if ai_score > 80: verdict = "🤖 Likely AI Generated"
        elif ai_score > 50: verdict = "❓ Mixed / Suspicious"
        else: verdict = "👤 Likely Human Written"

        return {
            "ai_probability_score": ai_score,
            "verdict": verdict,
            "avg_perplexity": round(avg_perplexity, 2),
            "burstiness_score": round(burstiness, 2),
            "sentence_count": len(sentences)
        }

# Initialize Detector (Global)
ai_detector = AIDetector()

In [ ]:
# ============================================================================
# ENHANCED PDF EXTRACTION WITH PYMUPDF (FITZ)
# ============================================================================

def extract_text_with_pymupdf(file_path: str) -> Dict:
    """Extract text and metadata from PDF using PyMuPDF (fitz)."""
    try:
        doc = fitz.open(file_path)

        result = {
            "text": "",
            "pages": [],
            "metadata": doc.metadata,
            "page_count": len(doc)
        }

        full_text = []

        for page_num in range(len(doc)):
            page = doc[page_num]
            page_text = page.get_text("text")
            full_text.append(page_text)

            page_info = {
                "page_number": page_num + 1,
                "text": page_text,
                "text_length": len(page_text)
            }

            result["pages"].append(page_info)

        result["text"] = "\n".join(full_text)

        doc.close()

        print(f"✅ PyMuPDF extracted {len(result['text'])} characters from {result['page_count']} pages")

        return result

    except Exception as e:
        print(f"❌ PyMuPDF error: {e}")
        return {"text": "", "pages": [], "metadata": {}, "page_count": 0}

# ============================================================================
# DOCLING INTEGRATION
# ============================================================================

def extract_with_docling(file_path: str) -> Dict:
    """Use Docling for advanced document parsing."""
    if not DOCLING_AVAILABLE:
        print("⚠️ Docling not available")
        return None

    try:
        print("🔄 Processing with Docling...")
        converter = DocumentConverter()

        result = converter.convert(file_path)

        docling_data = {
            "text": result.document.export_to_text(),
            "markdown": result.document.export_to_markdown(),
            "structure": str(result.document)
        }

        print(f"✅ Docling extracted {len(docling_data['text'])} characters")

        return docling_data

    except Exception as e:
        print(f"⚠️ Docling processing error: {e}")
        return None

# ============================================================================
# DOCX EXTRACTION
# ============================================================================

def extract_text_from_docx(file_path: str) -> str:
    """Extract text from DOCX with tables support."""
    try:
        doc = docx.Document(file_path)

        paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]

        table_text = []
        for table in doc.tables:
            for row in table.rows:
                row_text = [cell.text.strip() for cell in row.cells]
                table_text.append(' | '.join(row_text))

        full_text = '\n'.join(paragraphs + table_text)

        print(f"✅ DOCX extracted {len(full_text)} characters")

        return full_text

    except Exception as e:
        print(f"❌ DOCX error: {e}")
        return ""

# ============================================================================
# IMAGE OCR
# ============================================================================

def image_to_base64(image_path: str) -> str:
    """Convert image to base64."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def extract_text_from_image(image_path: str) -> str:
    """OCR using vision model."""
    base64_image = image_to_base64(image_path)

    image_type = "image/jpeg"
    if image_path.lower().endswith('.png'):
        image_type = "image/png"

    try:
        print("🔄 Running OCR with vision model...")
        response = client.chat.completions.create(
            model=OCR_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:{image_type};base64,{base64_image}"
                            }
                        },
                        {
                            "type": "text",
                            "text": "Extract all text from this CV/resume image. Preserve structure and formatting. Return only the text."
                        }
                    ]
                }
            ],
            max_tokens=3000
        )

        text = response.choices[0].message.content
        print(f"✅ OCR extracted {len(text)} characters")
        return text

    except Exception as e:
        print(f"❌ OCR error: {e}")
        print("⚠️ Trying alternative OCR model...")

        try:
            response = client.chat.completions.create(
                model="google/gemini-flash-1.5",
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "image_url",
                                "image_url": {
                                    "url": f"data:{image_type};base64,{base64_image}"
                                }
                            },
                            {
                                "type": "text",
                                "text": "Extract all visible text from this image. Return only the text content."
                            }
                        ]
                    }
                ],
                max_tokens=3000
            )

            text = response.choices[0].message.content
            print(f"✅ Alternative OCR extracted {len(text)} characters")
            return text

        except Exception as e2:
            print(f"❌ Alternative OCR failed: {e2}")
            return ""

# ============================================================================
# UNIFIED FILE PROCESSOR
# ============================================================================


def extract_links_from_pdf(file_path: str) -> Dict[str, list]:
    """
    Extract all hyperlinks from PDF pages.
    Returns dict with 'linkedin', 'github', 'others'.
    """
    try:
        doc = fitz.open(file_path)
        linkedin_links = []
        github_links = []
        other_links = []

        for page in doc:
            for link in page.get_links():
                uri = link.get("uri")
                if uri:

                    if "linkedin.com" in uri.lower():
                        if uri not in linkedin_links:
                            linkedin_links.append(uri)

                    elif "github.com" in uri.lower():
                        if uri not in github_links:
                            github_links.append(uri)
                    else:
                        if uri not in other_links:
                            other_links.append(uri)

        doc.close()

        if linkedin_links or github_links:
            print(f"🔗 Extracted links: LinkedIn={len(linkedin_links)}, GitHub={len(github_links)}")

        return {
            "linkedin": linkedin_links,
            "github": github_links,
            "others": other_links
        }

    except Exception as e:
        print(f"❌ Error extracting links from PDF: {e}")
        return {"linkedin": [], "github": [], "others": []}

#


def process_file(file_path: str) -> Tuple[str, Dict]:
    """
    Process any file type and extract text + hyperlinks.
    Returns: (text_content, links_dict)
    """
    file_extension = file_path.lower().split('.')[-1]
    links_dict = {"linkedin": [], "github": [], "others": []}

    if file_extension == 'pdf':
        print("📄 Processing PDF with PyMuPDF...")
        pymupdf_result = extract_text_with_pymupdf(file_path)
        text = pymupdf_result.get("text", "")


        links_dict = extract_links_from_pdf(file_path)

        return text, links_dict

    elif file_extension in ['docx', 'doc']:
        print("📝 Processing DOCX...")
        return extract_text_from_docx(file_path), links_dict

    elif file_extension in ['jpg', 'jpeg', 'png', 'bmp', 'tiff']:
        print("🖼️ Processing Image...")
        return extract_text_from_image(file_path), links_dict

    else:
        raise ValueError(f"Unsupported file type: {file_extension}")

In [ ]:
# ============================================================================
# DOCLING WITH LLM FOR JSON OUTPUT
# ============================================================================

def parse_cv_with_docling_llm(file_path: str) -> Dict:
    """
    Use Docling to extract text, then use LLM to convert to JSON.
    This ensures proper JSON structure while using Docling's extraction.
    """

    print("🔄 Step 1: Extracting text with Docling + PyMuPDF...")

    # Extract with PyMuPDF
    pymupdf_result = extract_text_with_pymupdf(file_path)
    text = pymupdf_result.get("text", "")

    # ====== ADD HYPERLINK EXTRACTION HERE ======
    links_dict = extract_links_from_pdf(file_path)

    if links_dict["linkedin"]:
        linkedin_text = "\n".join(links_dict["linkedin"])
        text += f"\n\n--- EXTRACTED LINKEDIN LINKS ---\n{linkedin_text}"

    if links_dict["github"]:
        github_text = "\n".join(links_dict["github"])
        text += f"\n\n--- EXTRACTED GITHUB LINKS ---\n{github_text}"

        # Also try Docling
    docling_result = None
    if DOCLING_AVAILABLE:
        docling_result = extract_with_docling(file_path)

    # Combine both sources
    combined_text = text
    if docling_result and docling_result.get("text"):
        combined_text = text + "\n\n--- DOCLING OUTPUT ---\n\n" + docling_result["text"]

    if not combined_text:
        print("❌ Failed to extract text!")
        return get_empty_cv_template_essentials()

    print(f"✅ Extracted {len(combined_text)} characters")

    # Step 2: Use LLM to parse into JSON (ESSENTIALS ONLY)
    print("🔄 Step 2: Converting to JSON with LLM (essentials only)...")

    if len(combined_text) > 10000:
        combined_text = combined_text[:10000]
        print("⚠️ Text truncated to 10k chars")

    prompt = f"""You are an expert CV parser. Extract ESSENTIAL information from this CV/resume.\n\nExtract ONLY these essential fields:\n1. Personal Info: full_name, email, phone, linkedin\n2. Professional Summary\n3. Skills (technical and soft skills)\n4. Work Experience\n\nIMPORTANT INSTRUCTIONS:\n- LinkedIn URL: Extract the actual hyperlink (full URL), NOT just the text "LinkedIn"\n- GitHub URL: Extract the actual hyperlink (full URL), NOT just the text "GitHub"\n- If hyperlinks are provided in the text sections (e.g., "--- EXTRACTED LINKEDIN LINKS ---"), use those URLs\n\nRETURN THIS EXACT JSON STRUCTURE:\n{{\n  "full_name": "candidate's full name",\n  "email": "email address",\n  "phone": "phone number",\n  "linkedin": "Full LinkedIn URL (e.g., https://linkedin.com/in/username)",\n  "github": "Full GitHub URL (e.g., https://github.com/username)",\n  "summary": "professional summary or objective",\n  "skills": {{\n    "technical": ["skill1", "skill2", "skill3"],\n    "soft": ["skill1", "skill2"]\n  }},\n  "experience": [\n    {{\n      "title": "job title",\n      "company": "company name",\n      "duration": "time period",\n      "description": "job description and responsibilities"\n    }}\n  ]\n}}\n\nCV TEXT:\n{combined_text}\n\nIMPORTANT: Return ONLY the JSON object. No explanations, no markdown, just pure JSON."""

    try:
        response = client.chat.completions.create(
            model=PARSING_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a CV parser. Extract essential information and return only valid JSON. No explanations, no markdown fences, just pure JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=3000,
            temperature=0.1
        )

        result = response.choices[0].message.content.strip()

        print(f"📝 Received {len(result)} characters from LLM")

        # Clean JSON
        if '```json' in result:
            result = result.split('```json')[1].split('```')[0]
        elif '```' in result:
            parts = result.split('```')
            if len(parts) >= 2:
                result = parts[1]

        result = result.strip()

        if not result.startswith('{'):
            start = result.find('{')
            end = result.rfind('}')
            if start != -1 and end != -1:
                result = result[start:end+1]

        parsed = json.loads(result)
        print("✅ Docling + LLM parsing successful!")

        return parsed

    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error: {e}")
        print(f"Response preview: {result[:300] if 'result' in locals() else 'N/A'}")

        # Try fallback model
        print(f"\n⚠️ Trying fallback model: {PARSING_FALLBACK}")
        try:
            response = client.chat.completions.create(
                model=PARSING_FALLBACK,
                messages=[
                    {"role": "system", "content": "Extract CV essentials and return only JSON. No explanations."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=3000,
                temperature=0.1
            )

            result = response.choices[0].message.content.strip()

            if '```json' in result:
                result = result.split('```json')[1].split('```')[0]
            elif '```' in result:
                parts = result.split('```')
                if len(parts) >= 2:
                    result = parts[1]

            result = result.strip()

            if not result.startswith('{'):
                start = result.find('{')
                end = result.rfind('}')
                if start != -1 and end != -1:
                    result = result[start:end+1]

            parsed = json.loads(result)
            print("✅ Fallback model succeeded!")
            return parsed

        except Exception as e2:
            print(f"❌ Fallback model failed: {e2}")

    except Exception as e:
        print(f"❌ Unexpected error: {e}")

    # Return empty template
    print("⚠️ Returning empty template")
    return get_empty_cv_template_essentials()

def get_empty_cv_template_essentials() -> Dict:
    """Return empty CV template with essentials only."""
    return {
        "full_name": "",
        "email": "",
        "phone": "",
        "linkedin": "",
        "summary": "",
        "skills": {
            "technical": [],
            "soft": []
        },
        "experience": []
    }

In [ ]:
# ============================================================================
# LLM-BASED CV PARSING (FULL DETAILS)
# ============================================================================

def parse_cv_with_llm(cv_text: str) -> Dict:
    """Parse CV using pure LLM - extracts ALL details."""

    if len(cv_text) > 10000:
        cv_text = cv_text[:10000]
        print("⚠️ Text truncated to 10k chars")

    prompt = f"""You are an expert CV parser. Extract ALL information from this CV/resume.\n\nEXTRACTION RULES:\n1. Find and extract EVERY piece of information.\n2. For contact info: email, phone, LinkedIn, GitHub, portfolio, location.\n3. For skills: categorize as technical, soft, languages, tools.\n4. For experience: extract ALL jobs with complete details.\n5. For education: extract ALL degrees with details.\n6. Extract certifications, projects, achievements.\n7. If ANY field is missing, use empty string "" or empty array [].\n\n8. Volunteer experience does NOT have to be explicitly labeled as "Volunteer".\n   Infer volunteer work from context such as:\n   - Student activities\n   - Community work\n   - NGOs or non-profit organizations\n   - Unpaid roles\n   - Organizing events, mentoring, teaching, helping initiatives\n   If the work is unpaid or community-based, include it under "volunteer".\n\n9. If "job_title" is not explicitly stated, infer the most appropriate job title\n   based on:\n   - Work experience\n   - Skills\n   - Projects\n   - Education\n   Use a standard, professional job title (e.g., "Software Engineer", "Frontend Developer", "Data Analyst").\n\n10. The "certifications" field may include:\n    - Certifications\n    - Online courses\n    - Training programs\n    - Bootcamps\n    - Workshops\n    Even if they are not explicitly called a "certificate".\n\n11. Do NOT include academic degrees (Bachelor, Master, PhD) in "certifications".\n\n12. Do NOT guess information without strong evidence in the CV.\n13. Inferred fields must be reasonable and aligned with the CV content.\n14. If inference is not possible, leave the field empty.\n15. IMPORTANT INSTRUCTIONS:\n- LinkedIn URL: Extract the actual hyperlink (full URL), NOT just the text "LinkedIn"\n- GitHub URL: Extract the actual hyperlink (full URL), NOT just the text "GitHub"\n- If hyperlinks are provided in the text sections (e.g., "--- EXTRACTED LINKEDIN LINKS ---"), use those URLs\n\nRETURN THIS EXACT JSON STRUCTURE:\n{{\n  "full_name": "candidate's full name",\n  "email": "email address",\n  "phone": "phone number with country code",\n  "linkedin": "Full LinkedIn URL (e.g., https://linkedin.com/in/username)",\n  "github": "Full GitHub URL (e.g., https://github.com/username)",\n  "job_title": "current or target job title",\n  "summary": "professional summary or objective statement",\n\n  "education": [\n    {{\n      "degree": "full degree name and major",\n      "institution": "university/college name",\n      "location": "city, country",\n      "start_date": "MM/YYYY",\n      "end_date": "MM/YYYY",\n      "gpa": "GPA if mentioned"\n    }}\n  ],\n  "skills": {{\n    "technical": ["Python", "JavaScript", "React", "etc"],\n    "soft": ["Leadership", "Communication", "etc"],\n    "tools": ["Git", "Docker", "AWS", "etc"]\n  }},\n  "experience": [\n    {{\n      "title": "exact job title",\n      "company": "company name",\n      "location": "city, country",\n      "employment_type": "Full-time/Part-time/Contract",\n      "start_date": "MM/YYYY",\n      "end_date": "MM/YYYY or Present",\n      "duration": "calculated duration",\n      "responsibilities": ["responsibility 1", "responsibility 2"],\n      "achievements": ["achievement 1", "achievement 2"],\n      "technologies": ["tech used 1", "tech used 2"]\n    }}\n  ],\n  "certifications": [\n    {{\n      "name": "Name of the certification, course, training, or program",\n      "issuer": "Issuing organization or platform (e.g., Coursera, Udemy, Google, university, company)",\n      "issue_date": "MM/YYYY"\n    }}\n  ],\n  "projects": [\n    {{\n      "name": "project name",\n      "description": "detailed description",\n      "technologies": ["tech 1", "tech 2"],\n      "date": "MM/YYYY"\n    }}\n  ],\n  "languages": [\n    {{\n      "language": "language name",\n      "proficiency": "Native/Fluent/Professional/Basic"\n    }}\n  ],\n  "awards": [\n    {{\n      "title": "award title",\n      "issuer": "organization",\n      "description": "award description"\n    }}\n  ],\n  "volunteer": [\n    {{\n      "role": "volunteer role",\n      "organization": "organization name",\n      "duration": "time period",\n      "description": "what you did"\n    }}\n  ]\n}}\n\nCV TEXT:\n{cv_text}\n\nIMPORTANT: Return ONLY the JSON object. No explanations, no markdown, just pure JSON."""

    try:
        print(f"🤖 Parsing with {PARSING_MODEL}...")

        response = client.chat.completions.create(
            model=PARSING_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a professional CV parser. You extract information and return only valid JSON. No explanations, no markdown fences, just pure JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=4000,
            temperature=0.1
        )

        result = response.choices[0].message.content.strip()

        print(f"📝 Received {len(result)} characters from LLM")

        if '```json' in result:
            result = result.split('```json')[1].split('```')[0]
        elif '```' in result:
            parts = result.split('```')
            if len(parts) >= 2:
                result = parts[1]

        result = result.strip()

        if not result.startswith('{'):
            start = result.find('{')
            end = result.rfind('}')
            if start != -1 and end != -1:
                result = result[start:end+1]

        parsed = json.loads(result)
        print("✅ CV parsing successful!")
        return parsed

    except Exception as e:
        print(f"❌ Error: {e}")

    print("⚠️ Returning empty template")
    return {
        "full_name": "",
        "email": "",
        "phone": "",
        "linkedin": "",
        "github": "",
        "job_title": "",
        "summary": "",
        "education": [],
        "skills": {"technical": [], "soft": [], "tools": []},
        "experience": [],
        "certifications": [],
        "projects": [],
        "languages": [],
        "awards": [],
        "volunteer": []
    }

In [ ]:
# ============================================================================
# LLM-BASED ATS ANALYSIS
# ============================================================================

def analyze_ats_with_llm(parsed_cv: Dict, cv_text: str) -> Dict:
    """Use LLM to analyze ATS compatibility with strict scoring criteria."""

    cv_summary = json.dumps(parsed_cv, indent=2)

    prompt = f"""You are a STRICT ATS (Applicant Tracking System) analyst and senior recruiter with 15+ years of experience.\n\nAnalyze this CV with RIGOROUS STANDARDS. Most CVs score between 50-75. Only exceptional CVs score above 85.\n\nCV DATA:\n{cv_summary}\n\nSTRICT SCORING CRITERIA:\n\n**FORMATTING (0-100):**\n- 100: Perfect ATS-friendly format, no tables/images, clear sections\n- 80-90: Good format with minor issues\n- 60-79: Readable but has formatting problems\n- 40-59: Poor formatting, ATS may fail to parse\n- 0-39: Completely unreadable by ATS\n\n**CONTENT QUALITY (0-100):**\n- 100: Every bullet uses strong action verbs + quantified results (e.g., "Increased revenue by 35%")\n- 80-90: Most bullets have metrics, clear impact\n- 60-79: Generic descriptions, few metrics\n- 40-59: Vague descriptions like "responsible for..."\n- 0-39: No measurable achievements\n\n**KEYWORD OPTIMIZATION (0-100):**\n- 100: Industry keywords naturally integrated 15+ times\n- 80-90: Good keyword usage (8-14 occurrences)\n- 60-79: Some keywords (4-7 occurrences)\n- 40-59: Very few keywords (1-3 occurrences)\n- 0-39: No relevant keywords\n\n**EXPERIENCE PRESENTATION (0-100):**\n- 100: Clear progression, quantified achievements, no gaps\n- 80-90: Good career story with most achievements quantified\n- 60-79: Timeline clear but lacks quantification\n- 40-59: Gaps or unclear progression\n- 0-39: No clear experience narrative\n\n**SKILLS RELEVANCE (0-100):**\n- 100: 15+ in-demand technical skills + soft skills\n- 80-90: 10-14 relevant skills\n- 60-79: 5-9 skills\n- 40-59: Generic or outdated skills\n- 0-39: No relevant skills listed\n\n**COMPLETENESS (0-100):**\n- 100: All sections (contact, summary, experience, education, skills, certifications)\n- 80-90: Missing 1 section\n- 60-79: Missing 2 sections\n- 40-59: Missing 3+ sections\n- 0-39: Bare minimum information\n\n**PROFESSIONALISM (0-100):**\n- 100: Perfect grammar, professional email, LinkedIn, no typos\n- 80-90: 1-2 minor issues\n- 60-79: 3-5 issues\n- 40-59: Multiple grammar/spelling errors\n- 0-39: Unprofessional presentation\n\n**CALCULATE OVERALL SCORE:**\nOverall Score = Average of all 7 categories\n\n**GRADE MAPPING:**\n- A+ (95-100): Elite, top 1%\n- A (90-94): Excellent\n- B+ (85-89): Very Good\n- B (80-84): Good\n- C+ (75-79): Above Average\n- C (70-74): Average\n- D (60-69): Below Average\n- F (<60): Needs Major Work\n\n**CRITICAL ISSUES (must flag):**\n- Missing contact information\n- No quantified achievements\n- Employment gaps > 6 months unexplained\n- Typos or grammar errors\n- Generic objective statements\n- Outdated skills\n\n**IMPORTANT CALIBRATION EXAMPLES:**\n\nExample 1 - Score: 45/100 (Grade F):\n- Missing email\n- No metrics in experience ("Worked on projects")\n- Skills: "Microsoft Office, Email"\n- No LinkedIn\n\nExample 2 - Score: 68/100 (Grade D):\n- Contact info present\n- Experience listed but no achievements\n- Generic skills\n- 2-3 typos\n\nExample 3 - Score: 82/100 (Grade B):\n- All sections complete\n- Some quantified achievements\n- Relevant skills\n- Minor formatting issues\n\nExample 4 - Score: 93/100 (Grade A):\n- Perfect format\n- Every role has 3+ quantified achievements\n- 15+ industry keywords\n- Professional LinkedIn/GitHub\n- No errors\n\nNOW ANALYZE THIS CV:\n{cv_summary}\n\nBE STRICT. BE HONEST. Most CVs need improvement.\n\nRETURN THIS EXACT JSON:\n{{\n  "overall_score": <calculated_average>,\n  "grade": "<letter_grade>",\n  "summary_feedback": "harsh but fair 2-sentence assessment",\n  "scores": {{\n    "formatting": <0-100>,\n    "content_quality": <0-100>,\n    "keyword_optimization": <0-100>,\n    "experience_presentation": <0-100>,\n    "skills_relevance": <0-100>,\n    "completeness": <0-100>,\n    "professionalism": <0-100>\n  }},\n  "strengths": [\n    "specific strength with evidence",\n    "another strength with evidence"\n  ],\n  "weaknesses": [\n    "critical weakness with impact",\n    "another weakness with impact"\n  ],\n  "critical_issues": [\n    "issue 1 (if any)",\n    "issue 2 (if any)"\n  ],\n  "improvement_suggestions": [\n    {{\n      "category": "category",\n      "priority": "Critical/High/Medium",\n      "suggestion": "actionable suggestion",\n      "impact": "expected score improvement (+X points)"\n    }}\n  ],\n  "missing_elements": ["element1", "element2"],\n  "keywords_analysis": {{\n    "strong_keywords": ["found1", "found2"],\n    "missing_keywords": ["missing1", "missing2"],\n    "suggested_keywords": ["add1", "add2"]\n  }},\n  "recruiter_perspective": "paragraph explaining hiring decision",\n  "next_steps": ["action1", "action2"]\n}}\n\nReturn ONLY JSON. NO markdown. Be brutally honest."""

    try:
        print(f"🤖 ATS analysis with {ATS_MODEL}...")

        response = client.chat.completions.create(
            model=ATS_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "You are a STRICT ATS analyst. Most CVs score 50-75. Only exceptional CVs score 85+. Be honest and critical. Return only valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            max_tokens=3000,
            temperature=0.2  # Lower temperature for more consistent scoring
        )

        result = response.choices[0].message.content.strip()

        # Clean JSON
        if '```json' in result:
            result = result.split('```json')[1].split('```')[0]
        elif '```' in result:
            parts = result.split('```')
            if len(parts) >= 2:
                result = parts[1]

        result = result.strip()

        if not result.startswith('{'):
            start = result.find('{')
            end = result.rfind('}')
            if start != -1 and end != -1:
                result = result[start:end+1]

        ats_result = json.loads(result)

        # Validation: Ensure score is realistic
        overall_score = ats_result.get('overall_score', 70)
        if overall_score > 95:
            print("⚠️ Score suspiciously high, recalibrating...")
            scores = ats_result.get('scores', {})
            avg = sum(scores.values()) / len(scores) if scores else 70
            ats_result['overall_score'] = round(avg)

        print(f"✅ ATS analysis complete! Score: {ats_result['overall_score']}/100")

        return ats_result

    except Exception as e:
        print(f"⚠️ ATS analysis error: {e}")

        # Fallback with more realistic defaults
        return {
            "overall_score": 65,
            "grade": "D+",
            "summary_feedback": "CV needs significant improvements in formatting and content quality.",
            "scores": {
                "formatting": 60,
                "content_quality": 55,
                "keyword_optimization": 50,
                "experience_presentation": 70,
                "skills_relevance": 65,
                "completeness": 75,
                "professionalism": 70
            },
            "strengths": ["CV structure is present"],
            "weaknesses": ["Lacks quantified achievements", "Missing keywords"],
            "critical_issues": ["No measurable results in experience section"],
            "improvement_suggestions": [
                {
                    "category": "Content",
                    "priority": "Critical",
                    "suggestion": "Add metrics to every achievement (e.g., 'Increased X by Y%')",
                    "impact": "+15 points"
                }
            ],
            "missing_elements": ["Quantified achievements"],
            "keywords_analysis": {
                "strong_keywords": [],
                "missing_keywords": ["Industry-specific terms"],
                "suggested_keywords": ["Add role-specific keywords"]
            },
            "recruiter_perspective": "CV shows potential but needs quantification and keyword optimization",
            "next_steps": ["Add metrics", "Optimize keywords", "Fix formatting"]
        }

In [ ]:
# ============================================================================
# LLM-BASED CV IMPROVEMENTS
# ============================================================================

def generate_improvements_with_llm(parsed_cv: Dict, ats_result: Dict) -> Dict:
    """Generate personalized improvement suggestions."""

    cv_summary = json.dumps(parsed_cv, indent=2)
    ats_summary = json.dumps(ats_result, indent=2)

    prompt = f"""You are a professional CV consultant and career coach.\n\nBased on this CV and ATS analysis, provide specific, actionable improvements.\n\nCV:\n{cv_summary}\n\nATS ANALYSIS:\n{ats_summary}\n\nPROVIDE DETAILED RECOMMENDATIONS IN JSON:\n{{\n  "priority_actions": [\n    {{\n      "action": "specific action to take",\n      "reason": "why this matters",\n      "how_to": "step-by-step how to implement",\n      "priority": "Critical/High/Medium"\n    }}\n  ],\n  "content_rewrites": {{\n    "professional_summary": "rewritten professional summary that's ATS-friendly and compelling",\n    "experience_improvements": [\n      {{\n        "current": "current description",\n        "improved": "improved version with action verbs and metrics",\n        "why_better": "explanation"\n      }}\n    ]\n  }},\n  "skills_strategy": {{\n    "skills_to_add": ["skill 1", "skill 2"],\n    "skills_to_emphasize": ["skill 1", "skill 2"],\n    "skills_to_remove": ["skill 1", "skill 2"],\n    "how_to_demonstrate": ["method 1", "method 2"]\n  }},\n  "formatting_checklist": [\n    "formatting tip 1",\n    "formatting tip 2"\n  ],\n  "keyword_strategy": {{\n    "must_add": ["keyword 1", "keyword 2"],\n    "contextual_usage": [\n      {{\n        "keyword": "keyword",\n        "where": "section",\n        "example": "how to use it"\n      }}\n    ]\n  }},\n  "achievement_framework": {{\n    "suggested_achievements": [\n      "achievement using STAR method",\n      "achievement with metrics"\n    ],\n    "quantification_tips": ["tip 1", "tip 2"]\n  }},\n  "30_day_plan": [\n    {{\n      "week": 1,\n      "tasks": ["task 1", "task 2"],\n      "goal": "what to achieve"\n    }}\n  ]\n}}\n\nReturn ONLY JSON:"""

    try:
        print(f"🤖 Generating improvements with {SCORING_MODEL}...")

        response = client.chat.completions.create(
            model=SCORING_MODEL,
            messages=[
                {"role": "system", "content": "You are a CV improvement expert. Provide actionable advice in JSON."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=3000,
            temperature=0.4
        )

        result = response.choices[0].message.content.strip()

        if '```json' in result:
            result = result.split('```json')[1].split('```')[0]
        elif '```' in result:
            parts = result.split('```')
            if len(parts) >= 2:
                result = parts[1]

        result = result.strip()

        if not result.startswith('{'):
            start = result.find('{')
            end = result.rfind('}')
            if start != -1 and end != -1:
                result = result[start:end+1]

        improvements = json.loads(result)
        print("✅ Improvements generated!")

        return improvements

    except Exception as e:
        print(f"⚠️ Improvements generation error: {e}")

        return {
            "priority_actions": [
                {
                    "action": "Review ATS feedback",
                    "reason": "Address identified issues",
                    "how_to": "Follow the suggestions provided",
                    "priority": "High"
                }
            ],
            "content_rewrites": {},
            "skills_strategy": {},
            "formatting_checklist": [],
            "keyword_strategy": {},
            "achievement_framework": {},
            "30_day_plan": []
        }

In [ ]:
# ============================================================================
# ADVANCED JOB MATCHING
# ============================================================================

class JobMatcher:
    """AI-powered job matching with multi-criteria analysis."""

    def __init__(self):
        self.model = embedding_model
        self.weights = {
            'job_title': 0.25,
            'skills': 0.30,
            'experience': 0.25,
            'summary': 0.10,
            'projects': 0.05,
            'certifications': 0.05
        }

    def create_cv_components(self, parsed_cv: Dict) -> Dict[str, str]:
        """Create separate text components for weighted matching."""
        components = {}

        if parsed_cv.get('job_title'):
            components['job_title'] = f"Target Role: {parsed_cv['job_title']}"
        else:
            components['job_title'] = ""

        skills_data = parsed_cv.get('skills', {})
        all_skills = []
        if isinstance(skills_data, dict):
            for category, skills_list in skills_data.items():
                if isinstance(skills_list, list):
                    all_skills.extend(skills_list)
        components['skills'] = f"Skills: {', '.join(all_skills)}" if all_skills else ""

        experience_parts = []
        for exp in parsed_cv.get('experience', []):
            exp_parts = []
            if exp.get('title'):
                exp_parts.append(exp['title'])
            if exp.get('company'):
                exp_parts.append(f"at {exp['company']}")
            if exp.get('description'):
                exp_parts.append(exp['description'])
            if exp.get('responsibilities'):
                if isinstance(exp['responsibilities'], list):
                    exp_parts.extend(exp['responsibilities'])
            if exp.get('achievements'):
                if isinstance(exp['achievements'], list):
                    exp_parts.extend(exp['achievements'])
            if exp_parts:
                experience_parts.append(' '.join(exp_parts))
        components['experience'] = ' '.join(experience_parts)

        components['summary'] = parsed_cv.get('summary', '')

        project_parts = []
        for proj in parsed_cv.get('projects', []):
            if proj.get('description'):
                project_parts.append(proj['description'])
        components['projects'] = ' '.join(project_parts)

        cert_parts = []
        for cert in parsed_cv.get('certifications', []):
            if cert.get('name'):
                cert_parts.append(f"Certified in {cert['name']}")
        components['certifications'] = ' '.join(cert_parts)

        return components

    def create_job_components(self, job: Dict) -> Dict[str, str]:
        """Create separate text components for job."""
        components = {}

        components['job_title'] = job.get('title', '')
        components['skills'] = job.get('required_skills', '') if isinstance(job.get('required_skills'), str) else ' '.join(job.get('required_skills', []))
        components['experience'] = job.get('description', '')
        components['summary'] = job.get('description', '')
        components['projects'] = ''
        components['certifications'] = job.get('preferred_qualifications', '') if isinstance(job.get('preferred_qualifications'), str) else ''

        return components

    def calculate_weighted_similarity(self, cv_components: Dict[str, str], job_components: Dict[str, str]) -> float:
        """Calculate weighted similarity across components."""
        total_score = 0.0
        total_weight = 0.0

        for component_name, weight in self.weights.items():
            cv_text = cv_components.get(component_name, '')
            job_text = job_components.get(component_name, '')

            if cv_text and job_text:
                cv_embedding = self.model.encode([cv_text])
                job_embedding = self.model.encode([job_text])

                similarity = cosine_similarity(cv_embedding, job_embedding)[0][0]
                total_score += similarity * weight
                total_weight += weight

        if total_weight > 0:
            return (total_score / total_weight) * 100
        return 0.0

    def match_with_llm(self, parsed_cv: Dict, job: Dict) -> Dict:
        """Use LLM for intelligent matching analysis."""

        cv_summary = json.dumps(parsed_cv, indent=2)
        job_summary = json.dumps(job, indent=2)

        prompt = f"""You are a recruitment matching expert.\n\nAnalyze how well this candidate matches this job position.\n\nCANDIDATE CV:\n{cv_summary}\n\nJOB POSITION:\n{job_summary}\n\nPROVIDE DETAILED MATCH ANALYSIS IN JSON:\n{{\n  "match_score": 85,\n  "match_level": "Excellent/Good/Fair/Poor",\n  "recommendation": "Apply immediately / Strong candidate / Consider / Not recommended",\n  "detailed_scores": {{\n    "skills_match": 90,\n    "experience_match": 85,\n    "education_match": 80,\n    "cultural_fit": 85,\n    "career_trajectory": 88\n  }},\n  "matched_requirements": [\n    "requirement 1",\n    "requirement 2"\n  ],\n  "missing_requirements": [\n    {{\n      "requirement": "missing skill/experience",\n      "importance": "Critical/High/Medium/Low",\n      "can_learn": true/false,\n      "time_to_acquire": "timeframe"\n    }}\n  ],\n  "strengths_for_role": [\n    "strength 1 specific to this role",\n    "strength 2 specific to this role"\n  ],\n  "concerns": [\n    "concern 1",\n    "concern 2"\n  ],\n  "differentiators": [\n    "what makes candidate stand out"\n  ],\n  "interview_talking_points": [\n    "point to emphasize in interview"\n  ],\n  "salary_competitiveness": "assessment",\n  "likelihood_of_offer": "High/Medium/Low",\n  "application_strategy": "specific advice for this application"\n}}\n\nReturn ONLY JSON:"""

        try:
            response = client.chat.completions.create(
                model=SCORING_MODEL,
                messages=[
                    {"role": "system", "content": "You are a recruitment expert. Analyze job matches and return JSON."},
                    {"role": "user", "content": prompt}
                ],
                max_tokens=2000,
                temperature=0.3
            )

            result = response.choices[0].message.content.strip()

            if '```json' in result:
                result = result.split('```json')[1].split('```')[0]
            elif '```' in result:
                parts = result.split('```')
                if len(parts) >= 2:
                    result = parts[1]

            result = result.strip()

            if not result.startswith('{'):
                start = result.find('{')
                end = result.rfind('}')
                if start != -1 and end != -1:
                    result = result[start:end+1]

            return json.loads(result)

        except Exception as e:
            print(f"⚠️ LLM matching error for {job.get('title', 'Unknown')}: {e}")
            return None

    def match_jobs(self, parsed_cv: Dict, jobs: List[Dict]) -> List[Dict]:
        """Match CV against jobs using weighted embeddings + LLM analysis."""

        print(f"🔍 Matching against {len(jobs)} jobs...")

        cv_components = self.create_cv_components(parsed_cv)

        results = []

        for job in jobs:
            print(f"  Analyzing: {job['title']}...")

            job_components = self.create_job_components(job)
            weighted_score = self.calculate_weighted_similarity(cv_components, job_components)

            llm_analysis = self.match_with_llm(parsed_cv, job)

            if llm_analysis:
                result = {
                    "job_title": job['title'],
                    "company": job.get('company', 'N/A'),
                    "location": job.get('location', 'N/A'),
                    "salary_range": job.get('salary_range', 'N/A'),
                    "semantic_similarity": round(weighted_score, 2),
                    **llm_analysis
                }
            else:
                result = {
                    "job_title": job['title'],
                    "company": job.get('company', 'N/A'),
                    "location": job.get('location', 'N/A'),
                    "salary_range": job.get('salary_range', 'N/A'),
                    "match_score": round(weighted_score, 2),
                    "match_level": self._get_match_level(weighted_score),
                    "recommendation": "Based on semantic similarity only"
                }

            results.append(result)

        results.sort(key=lambda x: x.get('match_score', 0), reverse=True)

        print("✅ Job matching complete!")

        return results

    def _get_match_level(self, score: float) -> str:
        """Get match level from score."""
        if score >= 85:
            return "🟢 Excellent"
        elif score >= 70:
            return "🟡 Good"
        elif score >= 55:
            return "🟠 Fair"
        else:
            return "🔴 Poor"

In [ ]:
# ============================================================================
# MAIN EXECUTION
# ============================================================================

def main(parsing_mode: str = "llm"):
    """
    Main execution flow.

    Args:
        parsing_mode: "docling" or "llm"
            - "docling": Use Docling+PyMuPDF extraction → LLM for JSON (essentials only)
            - "llm": Use PyMuPDF → LLM for JSON (full details, default)
    """

    print("=" * 80)
    print("🎯 JOBLENS - AI-POWERED CV ANALYSIS")
    print(f"📋 PARSING MODE: {parsing_mode.upper()}")
    print("=" * 80)
    print()

    print("📤 Upload your CV (PDF, DOCX, or Image)...")
    uploaded = files.upload()

    if not uploaded:
        print("❌ No file uploaded!")
        return

    file_name = list(uploaded.keys())[0]
    print(f"\n✅ File: {file_name}\n")

    # Parse CV based on mode
    print("=" * 80)
    if parsing_mode.lower() == "docling":
        print("STEP 1: DOCLING+PYMUPDF → LLM JSON (ESSENTIALS ONLY)")
        print("=" * 80)
        parsed_cv = parse_cv_with_docling_llm(file_name)
        cv_text = process_file(file_name)
    else:
        print("STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)")
        print("=" * 80)
        cv_text = process_file(file_name)

        if not cv_text:
            print("❌ Failed to extract text!")
            return

        print(f"✅ Extracted {len(cv_text)} characters")
        parsed_cv = parse_cv_with_llm(cv_text)

    print("\n📋 Parsed CV (JSON):")
    print(json.dumps(parsed_cv, indent=2, ensure_ascii=False))
    print()


    # AI Content Detection (NEW!)
    print("=" * 80)
    print("STEP 2: AI CONTENT DETECTION (CHEATING CHECK)")
    print("=" * 80)
    print("🕵️‍♂️ Analyzing CV for AI-generated content...\n")

    # Handle cv_text - could be string or tuple
    text_for_analysis = cv_text
    if isinstance(cv_text, tuple):
        text_for_analysis = cv_text[0] if cv_text else ""
    elif not isinstance(cv_text, str):
        text_for_analysis = str(cv_text)

    ai_result = ai_detector.analyze_text(text_for_analysis)

    print(f"🕵️‍♂️ AI DETECTION RESULTS:")
    print(f"  📊 AI Probability Score: {ai_result['ai_probability_score']}/100")
    print(f"  🎯 Verdict: {ai_result['verdict']}")
    print(f"  📈 Average Perplexity: {ai_result['avg_perplexity']}")
    print(f"  💥 Burstiness Score: {ai_result['burstiness_score']}")
    print(f"  📝 Sentences Analyzed: {ai_result['sentence_count']}")

    # Interpretation & Warnings
    if ai_result['ai_probability_score'] > 80:
        print(f"\n  🚨 WARNING: This CV shows strong indicators of AI-generated content!")
        print(f"     • The text is highly predictable and monotone")
        print(f"     • Consider requesting original work samples")
        print(f"     • Recommend deeper interview/technical assessment")
    elif ai_result['ai_probability_score'] > 50:
        print(f"\n  ⚠️  CAUTION: Some sections may contain AI-assisted content.")
        print(f"     • Mix of human and potentially AI-written text detected")
        print(f"     • Review carefully during interview process")
    else:
        print(f"\n  ✅ Content appears to be authentically human-written.")
        print(f"     • Natural language patterns detected")
        print(f"     • Good variation in sentence structure")
    print()


    # ATS Analysis
    print("=" * 80)
    print("STEP 2: COMPREHENSIVE ATS ANALYSIS")
    print("=" * 80)
    ats_result = analyze_ats_with_llm(parsed_cv, cv_text)

    print(f"\n📊 ATS SCORE: {ats_result['overall_score']}/100 ({ats_result['grade']})")
    print(f"💬 {ats_result['summary_feedback']}\n")

    print("📈 Detailed Scores:")
    for category, score in ats_result.get('scores', {}).items():
        print(f"  • {category.replace('_', ' ').title()}: {score}/100")

    print(f"\n✅ Strengths:")
    for s in ats_result.get('strengths', [])[:5]:
        print(f"  • {s}")

    print(f"\n⚠️ Weaknesses:")
    for w in ats_result.get('weaknesses', [])[:5]:
        print(f"  • {w}")

    if ats_result.get('critical_issues'):
        print(f"\n🚨 Critical Issues:")
        for issue in ats_result['critical_issues']:
            print(f"  • {issue}")

    print(f"\n💡 Top Improvements:")
    for imp in ats_result.get('improvement_suggestions', [])[:3]:
        print(f"  • [{imp.get('priority', 'N/A')}] {imp.get('suggestion', 'N/A')}")

    print(f"\n👔 Recruiter Perspective:")
    print(f"  {ats_result.get('recruiter_perspective', 'N/A')}")
    print()

    # Generate improvements
    print("=" * 80)
    print("STEP 3: PERSONALIZED IMPROVEMENT PLAN")
    print("=" * 80)
    improvements = generate_improvements_with_llm(parsed_cv, ats_result)

    print("\n🎯 Priority Actions:")
    for action in improvements.get('priority_actions', [])[:3]:
        print(f"\n  [{action.get('priority', 'N/A')}] {action.get('action', 'N/A')}")
        print(f"  Why: {action.get('reason', 'N/A')}")
        print(f"  How: {action.get('how_to', 'N/A')}")

    if improvements.get('content_rewrites', {}).get('professional_summary'):
        print(f"\n📝 Suggested Professional Summary:")
        print(f"  {improvements['content_rewrites']['professional_summary']}")

    if improvements.get('skills_strategy', {}).get('skills_to_add'):
        print(f"\n🔧 Skills to Add:")
        for skill in improvements['skills_strategy']['skills_to_add'][:5]:
            print(f"  • {skill}")
    print()

    # Job matching
    print("=" * 80)
    print("STEP 4: INTELLIGENT JOB MATCHING")
    print("=" * 80)

    sample_jobs = [
        {
            "title": "Senior Python Developer",
            "company": "Tech Innovations Ltd",
            "location": "Cairo, Egypt",
            "salary_range": "60k-90k EGP",
            "description": "We're seeking a Senior Python Developer with 5+ years of experience in backend development. Must have strong experience with Django/FastAPI, PostgreSQL, Redis, and cloud platforms (AWS/GCP). Experience with microservices architecture and Docker/Kubernetes is essential. You'll lead a team of 3-5 developers and work on high-scale fintech applications.",
            "required_skills": ["Python", "Django", "FastAPI", "PostgreSQL", "Redis", "AWS", "Docker", "Kubernetes", "Microservices"],
            "required_years": 5
        },
        {
            "title": "Machine Learning Engineer",
            "company": "AI Solutions Egypt",
            "location": "Remote",
            "salary_range": "80k-120k USD",
            "description": "Join our AI team to build cutting-edge ML solutions. We need someone with strong experience in deep learning, NLP, and computer vision. Must have production experience deploying ML models at scale. TensorFlow, PyTorch, and MLOps experience required.",
            "required_skills": ["Python", "TensorFlow", "PyTorch", "Machine Learning", "Deep Learning", "NLP", "Computer Vision", "MLOps", "Kubernetes"],
            "required_years": 3
        },
        {
            "title": "Full Stack Developer",
            "company": "Startup Hub",
            "location": "Alexandria, Egypt",
            "salary_range": "40k-60k EGP",
            "description": "Fast-growing startup needs a full stack developer. React + Node.js stack. You'll build features from scratch and own the full development cycle. Startup experience is a plus.",
            "required_skills": ["React", "Node.js", "JavaScript", "TypeScript", "MongoDB", "REST API", "Git"],
            "required_years": 2
        }
    ]

    matcher = JobMatcher()
    matches = matcher.match_jobs(parsed_cv, sample_jobs)

    print("\n🎯 JOB MATCHES:\n")

    for i, match in enumerate(matches, 1):
        print(f"{i}. {match.get('job_title', 'N/A')} at {match.get('company', 'N/A')}")
        print(f"   📍 {match.get('location', 'N/A')} | 💰 {match.get('salary_range', 'N/A')}")

        if match.get('detailed_scores'):
            print(f"   📊 Breakdown:")
            for key, val in match['detailed_scores'].items():
                print(f"      • {key.replace('_', ' ').title()}: {val}/100")

        if match.get('matched_requirements'):
            print(f"   ✅ Matched Requirements ({len(match['matched_requirements'])}):")
            for req in match['matched_requirements'][:3]:
                print(f"      • {req}")

        if match.get('missing_requirements'):
            print(f"   ❌ Missing Requirements ({len(match['missing_requirements'])}):")
            for miss in match['missing_requirements'][:3]:
                if isinstance(miss, dict):
                    print(f"      • {miss.get('requirement', 'N/A')} ({miss.get('importance', 'N/A')})")
                else:
                    print(f"      • {miss}")

        if match.get('strengths_for_role'):
            print(f"   💪 Your Strengths:")
            for strength in match['strengths_for_role'][:2]:
                print(f"      • {strength}")

        print(f"   💡 Recommendation: {match.get('recommendation', 'N/A')}")

        if match.get('application_strategy'):
            print(f"   📋 Strategy: {match['application_strategy']}")

        print()

    print("=" * 80)
    print("✅ ANALYSIS COMPLETE!")
    print("=" * 80)

    return {
        "parsed_cv": parsed_cv,
        "ats_result": ats_result,
        "improvements": improvements,
        "job_matches": matches
    }

In [ ]:
# ============================================================================
# INTERACTIVE RUNNER
# ============================================================================

def run_joblens():

    print("\n" + "=" * 80)
    print("🎯 JOBLENS - AI-POWERED CV ANALYSIS")
    print("=" * 80)
    print("\n📋 Choose CV processing mode:\n")
    print("1️⃣  Docling Mode - Extract basic CV information only")
    print("   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion")
    print("   └─ Best for: Fast analysis & essentials\n")

    print("2️⃣  LLM Mode - Full extraction of all CV details (Recommended)")
    print("   └─ Uses: PyMuPDF → LLM for JSON conversion")
    print("   └─ Best for: Comprehensive & detailed analysis\n")

    print("=" * 80)

    while True:
        choice = input("\n👉 Select mode (1 or 2): ").strip()

        if choice == "1":
            print("\n✅ Selected: Docling Mode (Basic extraction only)")
            parsing_mode = "docling"
            break
        elif choice == "2":
            print("\n✅ Selected: LLM Mode (Full analysis)")
            parsing_mode = "llm"
            break
        else:
            print("❌ Invalid choice! Please select 1 or 2")

    print("\n🚀 Starting...\n")

    result = main(parsing_mode=parsing_mode)

    return result

# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()

In [ ]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 1

✅ Selected: Docling Mode (Basic extraction only)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: DOCLING

📤 Upload your CV (PDF, DOCX, or Image)...


[INFO] 2026-01-30 18:59:19,591 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-30 18:59:19,593 [RapidOCR] device_config.py:57: Using GPU device with ID: 0


Saving Rana-Nasser-Mohamed-Resume.pdf to Rana-Nasser-Mohamed-Resume (11).pdf

✅ File: Rana-Nasser-Mohamed-Resume (11).pdf


🕵️‍♂️ Running AI Content Analysis (Perplexity & Burstiness)...
STEP 1: DOCLING+PYMUPDF → LLM JSON (ESSENTIALS ONLY)
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 4826 characters from 2 pages
🔗 Extracted links: LinkedIn=10, GitHub=5
🔄 Step 1: Extracting text with Docling + PyMuPDF...
✅ PyMuPDF extracted 4826 characters from 2 pages
🔗 Extracted links: LinkedIn=10, GitHub=5
🔄 Processing with Docling...


[INFO] 2026-01-30 18:59:19,674 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-30 18:59:19,676 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-30 18:59:20,118 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-30 18:59:20,121 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-01-30 18:59:20,131 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-01-30 18:59:20,135 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_infer.pth
[INFO] 2026-01-30 18:59:20,301 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-30 18:59:20,302 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-01-3

✅ Docling extracted 7094 characters
✅ Extracted 13455 characters
🔄 Step 2: Converting to JSON with LLM (essentials only)...
⚠️ Text truncated to 10k chars
📝 Received 2399 characters from LLM
✅ Docling + LLM parsing successful!
----------------------------------------
📊 AI Detection Results:
   • Verdict: 👤 Likely Human Written
   • AI Probability: 0%
   • Perplexity (Predictability): 470.7 (Lower = More AI-like)
   • Burstiness (Variation): 449.51 (Lower = More AI-like)
----------------------------------------

📋 Parsed CV (JSON):
{
  "full_name": "Rana Nasser Mohamed",
  "email": "rananasser760@gmail.com",
  "phone": "01225833467",
  "linkedin": "https://www.linkedin.com/in/rana-nasser-7b2375291",
  "github": "https://github.com/rananasser760",
  "summary": "Enthusiastic Computer Science student at Ain Shams University with a strong interest in Artificial Intelligence (AI), Data Engineering, and Physics. Seeking opportunities to apply technical skills and knowledge to innovative proje

In [ ]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 2

✅ Selected: LLM Mode (Full analysis)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: LLM

📤 Upload your CV (PDF, DOCX, or Image)...


Saving Mohamed_Amir_Resume.pdf to Mohamed_Amir_Resume (2).pdf

✅ File: Mohamed_Amir_Resume (2).pdf

STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 3490 characters from 1 pages
🔗 Extracted links: LinkedIn=1, GitHub=0
✅ Extracted 2 characters
🤖 Parsing with meta-llama/llama-3.3-70b-instruct...
📝 Received 3938 characters from LLM
✅ CV parsing successful!

📋 Parsed CV (JSON):
{
  "full_name": "Mohamed Amir",
  "email": "mohamedamiirshabeldin@gmail.com",
  "phone": "+20 1019779488",
  "linkedin": "https://www.linkedin.com/in/mohamed-shehab25/",
  "github": "",
  "job_title": "Junior Data Analyst",
  "summary": "Motivated and detail-oriented Junior Data Analyst with a strong academic foundation in Scientific Computing. Proficient in Python, SQL, Excel, and Power BI, with a passion for leveraging data to drive decision-making. Seeking an opportunity to utilize analytical skills and technical expertise to contribute to organizational success.",
 

In [ ]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 2

✅ Selected: LLM Mode (Full analysis)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: LLM

📤 Upload your CV (PDF, DOCX, or Image)...


Saving Mohamed_Saad_cv.pdf to Mohamed_Saad_cv.pdf

✅ File: Mohamed_Saad_cv.pdf

STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 2770 characters from 1 pages
🔗 Extracted links: LinkedIn=1, GitHub=1
✅ Extracted 2 characters
🤖 Parsing with meta-llama/llama-3.3-70b-instruct...
📝 Received 2366 characters from LLM
✅ CV parsing successful!

📋 Parsed CV (JSON):
{
  "full_name": "Mohamed Saad Mohamed",
  "email": "mohamed.saad.mohamed.464@gmail.com",
  "phone": "+20 3",
  "linkedin": "https://www.linkedin.com/in/mohamed-saad358/",
  "github": "https://github.com/MohamedSaad368",
  "job_title": "Machine Learning Practitioner",
  "summary": "Aspiring Machine Learning practitioner with a strong academic foundation in computer science and data analysis. Skilled in Python, NumPy, Pandas, and introductory ML algorithms. Passionate about applying machine learning techniques to solve real-world problems and eager to contribute to innovative projects while 

In [ ]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 1

✅ Selected: Docling Mode (Basic extraction only)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: DOCLING

📤 Upload your CV (PDF, DOCX, or Image)...


Saving AbdElrahman_ahmed_taher_cv.pdf to AbdElrahman_ahmed_taher_cv (1).pdf

✅ File: AbdElrahman_ahmed_taher_cv (1).pdf

STEP 1: DOCLING+PYMUPDF → LLM JSON (ESSENTIALS ONLY)
🔄 Step 1: Extracting text with Docling + PyMuPDF...
✅ PyMuPDF extracted 4235 characters from 2 pages
🔗 Extracted links: LinkedIn=1, GitHub=3
🔄 Processing with Docling...


[INFO] 2026-01-30 16:56:42,653 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-30 16:56:42,658 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-01-30 16:56:42,661 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.6.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-30 16:56:43,403 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-01-30 16:56:43,918 [RapidOCR] download_file.py:95: Successfully saved to: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-30 16:56:43,920 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_infer.pth
[INFO] 2026-01-30 16:56:44,150 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-01-30 16:56:44,151 [RapidOCR] device_config.py:57: Using GPU device with ID: 0
[INFO] 2026-01-30 16:56:44,152 [RapidOCR] download_file.py:68: Initiat

✅ Docling extracted 3709 characters
✅ Extracted 8224 characters
🔄 Step 2: Converting to JSON with LLM (essentials only)...
📝 Received 2437 characters from LLM
✅ Docling + LLM parsing successful!
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 4235 characters from 2 pages
🔗 Extracted links: LinkedIn=1, GitHub=3

📋 Parsed CV (JSON):
{
  "full_name": "Abdelrahman Ahmed Taher",
  "email": "abdelrahmaan.taheer@gmail.com",
  "phone": "01020793558",
  "linkedin": "https://www.linkedin.com/in/abdelrahman-taher",
  "github": "https://github.com/abdotaher123",
  "summary": "Final-year Computer Science student at Ain Shams University with a strong passion for Artificial Intelligence and cross-platform app development using Flutter. Completed a certified AI training program at Orange Digital Center and delivered machine learning workshops as an instructor at SFE. Proficient in Python, machine learning libraries, and mobile UI development. Seeking to contribute to innovative technology project

In [ ]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 2

✅ Selected: LLM Mode (Full analysis)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: LLM

📤 Upload your CV (PDF, DOCX, or Image)...


Saving Ahmad_Zo_ElFakkar_Resume 6.pdf to Ahmad_Zo_ElFakkar_Resume 6.pdf

✅ File: Ahmad_Zo_ElFakkar_Resume 6.pdf

STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 2807 characters from 1 pages
🔗 Extracted links: LinkedIn=1, GitHub=6
✅ Extracted 2 characters
🤖 Parsing with meta-llama/llama-3.3-70b-instruct...
📝 Received 4918 characters from LLM
✅ CV parsing successful!

📋 Parsed CV (JSON):
{
  "full_name": "Ahmad Zo ElFakkar",
  "email": "ahmadzoelfakkar@gmail.com",
  "phone": "+20 10 2205 2993",
  "linkedin": "https://linkedin.com/in/alandrio",
  "github": "https://github.com/AlandRio",
  "job_title": "Software Engineer",
  "summary": "Computer Science student specializing in Scientific Computing and Artificial Intelligence. I have been passionately programming since i was 12 years old. Skilled in building from scratch with minimal dependencies.",
  "education": [
    {
      "degree": "Scientific Computing - Computer Science",
      "institu

In [ ]:
# ============================================================================
# USAGE
# ============================================================================

result = run_joblens()


🎯 JOBLENS - AI-POWERED CV ANALYSIS

📋 Choose CV processing mode:

1️⃣  Docling Mode - Extract basic CV information only
   └─ Uses: Docling + PyMuPDF → LLM for JSON conversion
   └─ Best for: Fast analysis & essentials

2️⃣  LLM Mode - Full extraction of all CV details (Recommended)
   └─ Uses: PyMuPDF → LLM for JSON conversion
   └─ Best for: Comprehensive & detailed analysis


👉 Select mode (1 or 2): 2

✅ Selected: LLM Mode (Full analysis)

🚀 Starting...

🎯 JOBLENS - AI-POWERED CV ANALYSIS
📋 PARSING MODE: LLM

📤 Upload your CV (PDF, DOCX, or Image)...


Saving Mohamed Oraby Resume.pdf to Mohamed Oraby Resume.pdf

✅ File: Mohamed Oraby Resume.pdf

STEP 1: PYMUPDF → LLM JSON (FULL DETAILS)
📄 Processing PDF with PyMuPDF...
✅ PyMuPDF extracted 7987 characters from 3 pages
🔗 Extracted links: LinkedIn=1, GitHub=9
✅ Extracted 2 characters
🤖 Parsing with meta-llama/llama-3.3-70b-instruct...
📝 Received 8521 characters from LLM
✅ CV parsing successful!

📋 Parsed CV (JSON):
{
  "full_name": "Mohamed Oraby",
  "email": "mohamed.khalid.gamal@gmail.com",
  "phone": "+201208473203",
  "linkedin": "http://www.linkedin.com/in/mohamed-khalid-gamal",
  "github": "https://github.com/mohamed-khalid-gamal",
  "job_title": "Junior Full-stack Developer",
  "summary": "Passionate Computer Science undergraduate at Ain Shams University (Class of 2026, GPA: 3.7/4.0) with proven expertise in full-stack development. Currently interning as a .NET Engineer at Ejada Company. Recently achieved 10th place in the AIC2 Competition (68% accuracy) and 2nd place in ASU’s Ma